# Land cover classifier — Random Forest

**The question this notebook answers:** NDBI scores an IoU of **0.057** for built-up area. It calls desert sand "buildings". Does a classifier over the same features do better?

Every score here is reported next to the index rule it replaces, on the same pixels — so the comparison is like-for-like and any improvement is attributable to the model, not to better data.

| Class | Current rule | Measured IoU |
|---|---|---|
| water | MNDWI > −0.15 | 0.466 |
| vegetation | NDVI > 0.20 | 0.888 |
| **built-up** | **NDBI > −0.10** | **0.057** |
| bare | BSI > 0.15 | 0.747 |

**Runs anywhere** — no GPU. Random Forest is CPU-only, and the dataset is ~10,000 rows. Your laptop is fine; Colab and Kaggle also work.

**Input:** `evaluation/landcover_samples.csv`, produced by `backend/export_training_data.py` — Sentinel-2 pixels over Indian districts, labelled from ESA WorldCover.

In [ ]:
import json, pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Local run. In Colab or Kaggle, upload the CSV and set SAMPLES to it.
SAMPLES = Path("../evaluation/landcover_samples.csv")
MODELS = Path("../models")

FEATURES = ["B2", "B3", "B4", "B8", "B11", "B12",
            "ndvi", "ndwi", "mndwi", "ndbi", "ndmi", "bsi"]
CLASS_NAMES = {1: "water", 2: "vegetation", 3: "built-up", 4: "bare"}

# index, direction, threshold, IoU reported in surface.py
INDEX_RULES = {
    1: ("mndwi", "above", -0.15, 0.466),
    2: ("ndvi",  "above",  0.20, 0.888),
    3: ("ndbi",  "above", -0.10, 0.057),
    4: ("bsi",   "above",  0.15, 0.747),
}

frame = pd.read_csv(SAMPLES)
print(f"{len(frame):,} samples, {frame['district'].nunique()} districts")
frame.head()

## 1. Look at the data first

Before training anything, check the classes are actually separable. If built-up and bare overlap completely in every feature, no model will fix that.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
colours = {1: "#1f78b4", 2: "#33a02c", 3: "#e31a1c", 4: "#d8b365"}

for ax, index in zip(axes.ravel(), ["ndvi", "ndwi", "mndwi", "ndbi", "ndmi", "bsi"]):
    for class_id, name in CLASS_NAMES.items():
        values = frame.loc[frame["label"] == class_id, index]
        ax.hist(values, bins=50, alpha=0.5, label=name, color=colours[class_id])
    ax.set_title(index.upper())
    ax.set_yticks([])

    for class_id, (idx, _, threshold, _) in INDEX_RULES.items():
        if idx == index:
            ax.axvline(threshold, color="black", linestyle="--", linewidth=1.2)
            ax.text(threshold, ax.get_ylim()[1] * 0.92, f" rule: {threshold}",
                    fontsize=8)

axes[0, 0].legend(fontsize=8)
plt.suptitle("Where the classes sit in each index (dashed line = current threshold)")
plt.tight_layout()
plt.show()

print("Look at NDBI. If built-up and bare overlap heavily there, that is exactly")
print("why a single threshold on it scores 0.057 - one cut cannot separate them.")

## 2. Split by district, not at random

**This is the most important cell in the notebook.**

Two pixels twenty metres apart are nearly identical. A random split puts near-duplicates in both training and test, and reports an accuracy the model has not earned. This is the commonest way land cover results are overstated.

So whole districts are held out. **Jaisalmer and Bangalore Urban** specifically — because NDBI scored 0.001 in Jaisalmer and 0.322 in Bangalore, and holding out the easy districts would flatter everything.

In [ ]:
HOLDOUT = ["Jaisalmer", "Bangalore Urban"]

train = frame[~frame["district"].isin(HOLDOUT)]
test = frame[frame["district"].isin(HOLDOUT)]

x_train, y_train = train[FEATURES].to_numpy(), train["label"].to_numpy()
x_test, y_test = test[FEATURES].to_numpy(), test["label"].to_numpy()

print(f"train {len(train):>6,}  {sorted(set(train['district']))}")
print(f"test  {len(test):>6,}  {HOLDOUT}")
print("\nThe model will be scored on landscapes it has never seen.")

## 3. Train

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
model.fit(x_train, y_train)
predicted = model.predict(x_test)

print(f"Trained on {len(x_train):,} samples, {len(FEATURES)} features.")
print(f"scikit-learn {sklearn.__version__}")

## 4. The comparison that matters

Classifier against the index rule it replaces, on the same held-out pixels.

In [ ]:
def iou(pred, actual):
    tp = int(np.sum(pred & actual))
    fp = int(np.sum(pred & ~actual))
    fn = int(np.sum(~pred & actual))
    return tp / (tp + fp + fn) if (tp + fp + fn) else 0.0

def prec_rec(pred, actual):
    tp = int(np.sum(pred & actual)); fp = int(np.sum(pred & ~actual))
    fn = int(np.sum(~pred & actual))
    return (tp / (tp + fp) if tp + fp else 0.0,
            tp / (tp + fn) if tp + fn else 0.0)

rows = []
for class_id, name in CLASS_NAMES.items():
    actual = y_test == class_id
    if not actual.any():
        continue

    rf_mask = predicted == class_id
    rf_i = iou(rf_mask, actual)
    rf_p, rf_r = prec_rec(rf_mask, actual)

    index, _, threshold, reported = INDEX_RULES[class_id]
    rule_mask = test[index].to_numpy() > threshold
    rule_i = iou(rule_mask, actual)
    rule_p, rule_r = prec_rec(rule_mask, actual)

    rows.append({
        "class": name,
        "rule": f"{index.upper()} > {threshold}",
        "rule IoU": round(rule_i, 3),
        "RF IoU": round(rf_i, 3),
        "change": round(rf_i - rule_i, 3),
        "RF prec": round(rf_p, 3),
        "RF rec": round(rf_r, 3),
    })

comparison = pd.DataFrame(rows)
display(comparison)

built = comparison[comparison["class"] == "built-up"]
if not built.empty:
    change = built["change"].iloc[0]
    verdict = "beats" if change > 0 else "does NOT beat"
    print(f"\nBuilt-up: the classifier {verdict} the threshold ({change:+.3f}).")
    if change <= 0:
        print("Report that. A negative result on a spatial holdout is still a result,")
        print("and it is more defensible than a positive one on a random split.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(comparison))
width = 0.38

ax.bar(x - width/2, comparison["rule IoU"], width, label="index threshold", color="#94a3b8")
ax.bar(x + width/2, comparison["RF IoU"], width, label="random forest", color="#2a9d8f")

for i, row in comparison.iterrows():
    ax.text(i - width/2, row["rule IoU"] + 0.01, f"{row['rule IoU']:.3f}",
            ha="center", fontsize=9)
    ax.text(i + width/2, row["RF IoU"] + 0.01, f"{row['RF IoU']:.3f}",
            ha="center", fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(comparison["class"])
ax.set_ylabel("IoU"); ax.set_ylim(0, 1.05)
ax.set_title("Held-out districts: Jaisalmer and Bangalore Urban")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 5. Where it still gets confused

The interesting question is whether it still mistakes sand for concrete — the failure that made NDBI unusable.

In [ ]:
labels = sorted(CLASS_NAMES)
matrix = confusion_matrix(y_test, predicted, labels=labels)
normalised = matrix / matrix.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(normalised, cmap="Blues", vmin=0, vmax=1)

names = [CLASS_NAMES[l] for l in labels]
ax.set_xticks(range(len(labels))); ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(names)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{matrix[i, j]:,}\n{normalised[i, j]:.0%}",
                ha="center", va="center", fontsize=9,
                color="white" if normalised[i, j] > 0.5 else "black")

ax.set_title("Confusion on held-out districts")
plt.tight_layout(); plt.show()

print(classification_report(y_test, predicted, labels=labels,
                            target_names=names, digits=3, zero_division=0))

## 6. What the model actually uses

If NDBI dominates, the model is doing what the threshold did, only softer. If it spreads across bands, it has found something a single index cannot express — which is the argument for having it.

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colours = ["#2a9d8f" if f in ("ndvi", "ndwi", "mndwi", "ndbi", "ndmi", "bsi")
           else "#94a3b8" for f in importance.index]
ax.barh(importance.index, importance.values, color=colours)
ax.set_xlabel("importance"); ax.set_title("Feature importance (green = index, grey = raw band)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(importance.sort_values(ascending=False).head(5).to_string())

## 7. Save the model

With metadata. A bare `.pkl` is unreadable in three months, and scikit-learn pickles break across versions — the version is recorded so the backend can pin it.

In [ ]:
MODELS.mkdir(parents=True, exist_ok=True)

with (MODELS / "landcover_rf_v1.pkl").open("wb") as fh:
    pickle.dump(model, fh)

metadata = {
    "name": "landcover_rf_v1",
    "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "algorithm": "RandomForestClassifier",
    "n_estimators": 300,
    "sklearn_version": sklearn.__version__,
    "features_in_order": FEATURES,
    "classes": CLASS_NAMES,
    "training_samples": int(len(train)),
    "training_districts": sorted(set(train["district"])),
    "holdout_districts": HOLDOUT,
    "split": "spatial: whole districts held out",
    "results": comparison.to_dict(orient="records"),
}
(MODELS / "landcover_rf_v1.json").write_text(json.dumps(metadata, indent=2))

print("Saved to models/")
print(f"Pin scikit-learn=={sklearn.__version__} in backend/requirements.txt")

## What to do with the result

1. **If built-up improved a lot** — wire the model into `surface.py` behind the existing interface, update the reliability from `poor` to whatever it now measures, and remove the warning banner for that analysis.
2. **If it improved a little** — say so honestly. A move from 0.06 to 0.20 is still a broken method; the warning stays.
3. **If it did not improve** — that is a finding, not a failure. It would mean the features themselves cannot separate concrete from sand at 10 m, and the fix is better inputs (texture, seasonality, SAR) rather than a better classifier.
4. **Whatever happens**, the comparison table and the confusion matrix are figures for Chapter 7, and the spatial holdout is what makes them credible.

Do not switch to a random split to get a nicer number. The whole value of this result is that it was measured on districts the model never saw.